<a href="https://colab.research.google.com/github/Zheleikina/ML/blob/main/%D0%9B%D0%A09_%D0%96%D0%B5%D0%BB%D0%B5%D0%B9%D0%BA%D1%96%D0%BD%D0%B0_3_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Желейкіна А. С.  
**Група:** 3-9  
**Дата:** 30.10.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
import sys
print(sys.version)

3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]


In [2]:
%pip install --only-binary=:all: scikit-surprise==1.1.5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 8.6 MB/s  0:00:00


In [4]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [5]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [7]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [8]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:

# Кількість унікальних користувачів
print('Кількість користувачів:', ratings_df['user_id'].nunique())

# Кількість унікальних фільмів
print('Кількість фільмів:', ratings_df['item_id'].nunique())

# Мінімальне, максимальне, середнє значення rating
print('Мінімальна оцінка:', ratings_df['rating'].min())
print('Максимальна оцінка:', ratings_df['rating'].max())
print('Середня оцінка:', round(ratings_df['rating'].mean(), 2))

# Перевірка пропущених значень
print('Пропущені значення:')
print(ratings_df.isnull().sum())

print('Розподіл оцінок:')
print(ratings_df['rating'].value_counts().sort_index())

Кількість користувачів: 943
Кількість фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.53
Пропущені значення:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64
Розподіл оцінок:
rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64


**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._

У наборі даних є 943 користувачі та 1682 фільми. Користувачі поставили оцінки від 1 до 5, середня оцінка становить 3,53. Найчастіше зустрічається оцінка 4.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [9]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:

from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

print('Розмір навчальної вибірки:', train_df.shape)
print('Розмір тестової вибірки:', test_df.shape)

Розмір навчальної вибірки: (80000, 4)
Розмір тестової вибірки: (20000, 4)


In [10]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:

from surprise import Reader, Dataset

reader = Reader(rating_scale=data.reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

testset = list(
    test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None)
)

print('Кількість записів у train:', len(train_df))
print('Кількість записів у test:', len(testset))

Кількість записів у train: 80000
Кількість записів у test: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [11]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

Для SVD перевіряються кількість факторів, швидкість навчання та регуляризація, оскільки вони впливають на точність моделі й перенавчання.

Для KNNBasic перевіряються кількість сусідів, мінімальна кількість сусідів і метрика схожості, щоб знайти найкращий варіант для прогнозування оцінок.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [12]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:

from surprise import SVD
from surprise.model_selection import GridSearchCV

gs_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

gs_svd.fit(train_data)

print('Найкращі параметри RMSE:', gs_svd.best_params['rmse'])
print('Найкращі параметри MAE:', gs_svd.best_params['mae'])

Найкращі параметри RMSE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
Найкращі параметри MAE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}


In [13]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:

from surprise import KNNBasic

gs_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)

gs_knn.fit(train_data)

print('Найкращі параметри RMSE:', gs_knn.best_params['rmse'])
print('Найкращі параметри MAE:', gs_knn.best_params['mae'])

Найкращі параметри RMSE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}
Найкращі параметри MAE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


In [14]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:

print('SVD:')
print('Найкращий RMSE:', gs_svd.best_score['rmse'])
print('Параметри RMSE:', gs_svd.best_params['rmse'])
print('Найкращий MAE:', gs_svd.best_score['mae'])
print('Параметри MAE:', gs_svd.best_params['mae'])

print('\nKNNBasic:')
print('Найкращий RMSE:', gs_knn.best_score['rmse'])
print('Параметри RMSE:', gs_knn.best_params['rmse'])
print('Найкращий MAE:', gs_knn.best_score['mae'])
print('Параметри MAE:', gs_knn.best_params['mae'])

SVD:
Найкращий RMSE: 0.9422106153321694
Параметри RMSE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
Найкращий MAE: 0.7428015433604108
Параметри MAE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

KNNBasic:
Найкращий RMSE: 1.0268305192980343
Параметри RMSE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}
Найкращий MAE: 0.814493414139608
Параметри MAE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|---|---:|---:|---|
| SVD | 0.9422 | 0.7428 | n_factors=50, lr_all=0.01, reg_all=0.05 |
| KNNBasic | 1.0268 | 0.8145 | k=40, min_k=1, cosine, user_based=True |

SVD показала кращий результат, оскільки її RMSE та MAE нижчі, ніж у KNNBasic. Для SVD найкращими стали 50 факторів, швидкість навчання 0.01 і регуляризація 0.05. Для KNNBasic найкращими стали 40 сусідів, мінімум 1 сусід і метрика cosine.



## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [15]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:

# 6.1
best_svd_params = gs_svd.best_params['rmse']
best_knn_params = gs_knn.best_params['rmse']

# 6.2
trainset = train_data.build_full_trainset()

# 6.3
best_svd = SVD(**best_svd_params, random_state=42)
best_knn = KNNBasic(**best_knn_params)

best_svd.fit(trainset)
best_knn.fit(trainset)

print('Моделі успішно навчені.')

Computing the cosine similarity matrix...
Done computing similarity matrix.
Моделі успішно навчені.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [16]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:

svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print('SVD:')
print('RMSE:', svd_rmse)
print('MAE:', svd_mae)

print('\nKNNBasic:')
print('RMSE:', knn_rmse)
print('MAE:', knn_mae)

SVD:
RMSE: 0.9191746414249822
MAE: 0.7233554064545319

KNNBasic:
RMSE: 1.0119527444069378
MAE: 0.7984207578945732


In [17]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:

results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
})

results = results.sort_values(by='RMSE')

display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._


| Модель | Test RMSE | Test MAE |
|---|---:|---:|
| SVD | 0.9192 | 0.7234 |
| KNNBasic | 1.0120 | 0.7984 |

### Порівняння з CV

Результати на тестовій вибірці трохи кращі, ніж під час кросвалідації, але суттєво не відрізняються. SVD в обох випадках показала кращу якість, ніж KNNBasic.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [18]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:

# 8.1
if gs_svd.best_score['rmse'] <= gs_knn.best_score['rmse']:
    best_model = best_svd
    print("Обрана модель: SVD")
else:
    best_model = best_knn
    print("Обрана модель: KNNBasic")

# 8.2
try:
    inner_uid = trainset.to_inner_uid(user_id)

    # 8.3
    rated_items = set(
        iid for iid, rating in trainset.ur[inner_uid]
    )

    all_items = range(trainset.n_items)
    unrated_items = [
        trainset.to_raw_iid(inner_iid)
        for inner_iid in all_items
        if inner_iid not in rated_items
    ]

    # 8.4
    predictions = [
        (iid, best_model.predict(user_id, iid).est)
        for iid in unrated_items
    ]

    top_10 = sorted(
        predictions,
        key=lambda x: x[1],
        reverse=True
    )[:10]

    recommendations = pd.DataFrame(
        top_10,
        columns=['item_id', 'predicted_rating']
    )

    display(recommendations)

except ValueError:
    print(f"Користувача з ID {user_id} немає у навчальній вибірці.")

Обрана модель: SVD


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [x] Дані `ml-100k` завантажено та перевірено.
- [x] Є розбиття 80/20 з фіксованим `random_state`.
- [x] `Reader`, `train_data` і `testset` створені коректно.
- [x] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [x] Для SVD і KNNBasic показано параметри та результати CV.
- [x] Моделі навчено на повній навчальній частині.
- [x] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [x] Наведено порівняльну таблицю та інтерпретацію.
- [x] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [x] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** У ході лабораторної роботи було реалізовано та порівняно моделі SVD і KNNBasic. Кращий результат показала SVD з RMSE 0,9192 та MAE 0,7234 на тестовій вибірці. Для KNNBasic ці значення становили 1,0120 та 0,7984 відповідно. Результати тестування майже не відрізнялися від кросвалідації. Також було сформовано топ-10 рекомендацій для користувача на основі прогнозованих оцінок. Основним обмеженням є те, що моделі погано працюють із новими користувачами та фільмами, для яких ще немає оцінок.

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.